In [80]:
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [81]:
with open('input.txt', 'r') as f: 
    text = f.read()

In [82]:
print('length of text:', len(text))

length of text: 1115394


In [83]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [84]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [85]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [86]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [87]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
print(f'Using device: {device}')

Using device: cuda


In [88]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [89]:
block_size = 8
print(train_data[:block_size+1])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [90]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context} the target is {target}')

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [91]:
torch.manual_seed(1337)
batch_size = 4 
block_size = 8
def get_batch(split):
    data = train_data if split == 'train' else val_data 
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y 
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)
print('---')
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f'when input is {context.tolist()} the target is: {target}')

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')
---
when input is [24] the target is: 43
when input is [24, 43] the target is: 58
when input is [24, 43, 58] the target is: 5
when input is [24, 43, 58, 5] the target is: 57
when input is [24, 43, 58, 5, 57] the target is: 1
when input is [24, 43, 58, 5, 57, 1] the target is: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is: 39
when input is [44] the target is: 53
when input is [44, 53] the target is: 56
when input is [44, 53, 56] the target is: 1
when input is [44, 53, 56, 1] the target is: 58
w

In [92]:
print(xb)  # [B, T] = [4, 8]

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')


In [93]:
import torch 
import torch.nn as nn  
from torch.nn import functional as F 
torch.manual_seed(1337)
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size, device=device)
    def forward(self, idx, targets=None): # idx, targets (B,T)
        # B: batch size, T: time, C: class
        logits = self.token_embedding_table(idx) # (B,T,C) = (4, 8, 65)
        if targets is None:
            loss = None 
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss 
    def generate(self, idx, max_new_tokens):
        # idx: (B,T)
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # (B,T,C)
            logits = logits[:, -1, :] # (B,C)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx = torch.cat((idx, idx_next), dim=1) # (B,T+1)
        return idx
m = BigramLanguageModel(vocab_size)
out, loss = m(xb, yb)
print(out.shape)
print(loss)
idx = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(idx=idx, max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8573, device='cuda:0', grad_fn=<NllLossBackward0>)

JrTxbDkRZkNwc.wj,ZTxO-On-y$WK
baqPe?kMBFeA$G:XZSGgO-3cjMGd?gLhaGhX'YVX3tpgfNuwq&$WWv.tbaF :X3!FHaGeN


In [94]:
# typical learning rate for adam is 3e-4, here we use 1e-3 for smaller model
optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)

In [95]:
batch_size = 32 
import time
start = time.time()
for steps in range(1000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
elpased = time.time() - start 
print(f'{elpased:.2f}s')
print(loss.item())

2.27s
3.7310585975646973


In [96]:
idx = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(idx=idx, max_new_tokens=1000)[0].tolist()))


Fw:IQPryjf,fDEdnVzxzIINQh
Yow&$FmoofC,?-iB!HgV!?W;KuilW!cCe3 ixweYEYBgcie lxP;HFGEdkov EsSXZ3;qGk p.ghDW.FeWjbO!welJrjndiU;Qu.Clhs;ZqPnchXZY:?UBg;Mais'sl$wty.MJlv$nUHI.GBf?peYeigClE$nk:
GlqwTdlmotOSklmoaW-CQu&LhJVCeiib3TQeyjtG&dM$WZqETxe Ffx&$FjO,-LOKlAN-xT3H
XQLkOSmnvcruf?!A 'an;3;QD WRZ:fEE,gey$flPUE$FpYvMFEPrD?d KwSo .UyHK-NLba!a3,
yb&i-&
:adsabW!?!?EuD mYBvdinVcN'MKDyr?.nyzvr-lts'I&cc VEd!?-JknnNEYFPPk;RIKYy$rYBX
'WOSGIOShm 
bByB;E$WsVB-O?KwmNp&qqexleYmrdaJRY;qH-zabjOnVGCnnclqugUbxIE.TQ
muc$WUyX'y&;zvozvcYs u.m 
IEnassGJtlMc$Wjjk.xseueXjPeAbexlmovovLKu?dBa3P3LQUMPe FHMI-vbyKHKie :XSumo usGxVMFeRH,X'Ls:yibctGClmo;xcPun?i$YUDn3lQuMUtvo,Qmoxlone.hJUErTbaXkHgO,gFHTc-Eu
Nwwam k !?.vD bumlonF', MqGlNUllclovoPrivy,UdctvodISGurV?
XHgiKu
bHqZF&XnGIn'izXC,?-Fymf?HJIQzHoYCriuGoKtLoCnisevce u3OKN FRpDrtGOon wMb.C$Wg i-tO:$ukD OEYBlWRlbkH,UpPUs YMO, t:
HMb.cTjwatch&LivYoWoZt
Xr O:Iu;qyoCkg;gioCe?TU3tanBy.
uw:A.gflQiKWcQuwHRnk;&g.PSdMRB&cVHvQe,LhseX LPfla;oTElmo:CKayeo IsuZ
!?
mVf,Fme&wvvmiSjs 